# ModernBERT smell-token classifier training (46 routed heads)

Notebook-first training workflow for the `role_section_excerpts*.jsonl` dataset.

In [5]:
from pathlib import Path
import os
import json
import torch

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / 'training_inference').exists():
            return candidate
    raise RuntimeError('Could not locate repo root containing training_inference/.')

REPO_ROOT = _find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
print('REPO_ROOT =', REPO_ROOT)

from training_inference.data_utils import (
    create_dataset_build, create_tokenizer, PretokenizedSmellDataset, TokenClassificationCollator, summarize_dataset
)
from training_inference.model_pipeline import SharedEncoderSmellHeads, build_optimizer
from training_inference.train_loop import set_global_seed, make_dataloader, train_model


REPO_ROOT = C:\Users\jinmo\Documents\Github\encoder-bio\master-thesis-materials\pyexamine


In [6]:
# ==== User-editable settings (simple variables, no heavy config system) ====
SEED = 42
BACKBONE_NAME = 'answerdotai/ModernBERT-base'  # or ModernBERT-large if you have enough VRAM
MAX_LENGTH = 8192

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
NEG_POS_RATIO = 2.0        # None disables downsampling. Example: 1.0 (1:1), 2.0 (2:1)
BALANCE_PER_SMELL = False  # True = apply ratio per smell bucket

BATCH_SIZE_TRAIN = 1       # long context model; start conservative
BATCH_SIZE_EVAL = 1
NUM_EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
DROPOUT = 0.1
GRAD_ACCUM_STEPS = 8
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 0
USE_AMP = True

# Optional: class weights for routed token CE loss (helps with O dominance)
USE_CLASS_WEIGHTS = False
O_WEIGHT = 0.25

OUTPUT_DIR = REPO_ROOT / 'training_inference' / 'runs' / 'modernbert_routed_46h'
RESUME_CHECKPOINT = None  # e.g., OUTPUT_DIR / 'checkpoints' / 'epoch_001.pt'


In [7]:
# ==== Determinism ====
set_global_seed(SEED, deterministic=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


device = cpu


In [8]:
# ==== Load dataset / splits / mappings ====
dataset_build = create_dataset_build(
    repo_root=REPO_ROOT,
    preferred_dataset_name='role_section_excerpts.line.jsonl',
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
    train_neg_pos_ratio=NEG_POS_RATIO,
    balance_per_smell=BALANCE_PER_SMELL,
    split_manifest_dir=OUTPUT_DIR / 'split_manifests',
)

print('Train:', summarize_dataset(dataset_build.train_examples))
print('Val  :', summarize_dataset(dataset_build.val_examples))
print('Test :', summarize_dataset(dataset_build.test_examples))
print('num labels:', len(dataset_build.label_maps.label_to_id))
print('num smell heads:', len(dataset_build.smell_maps.smell_to_head))
assert len(dataset_build.smell_maps.smell_to_head) == 46, 'Expected 46 smells/heads.'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'metadata' / 'label_to_id.json').write_text(
    json.dumps(dataset_build.label_maps.label_to_id, ensure_ascii=False, indent=2), encoding='utf-8'
)
(OUTPUT_DIR / 'metadata' / 'smell_to_head.json').write_text(
    json.dumps(dataset_build.smell_maps.smell_to_head, ensure_ascii=False, indent=2), encoding='utf-8'
)


Train: {'num_examples': 462, 'positives': 154, 'negatives': 308, 'num_smells': 46, 'smell_names': ['Alternative Classes with Different Interfaces', 'Cyclic Dependency', 'Data Class', 'Data Clumps', 'Dead Code', 'Deep Inheritance Tree (DIT)', 'Duplicate Code', 'Excessive Comments', 'Feature Envy', 'God Object', 'High Coupling Between Object Classes (CBO)', 'High Cyclomatic Complexity', 'High Fan-in', 'High Fan-out', 'High Lack of Cohesion of Methods (LCOM)', 'High Lines of Code (LOC)', 'High Message Passing Coupling (MPC)', 'High Number of Classes per Module', 'High Number of Methods (NOM)', 'High Number of classes per Project', 'High Response for a Class (RFC)', 'High Weight of a Class (WAC)', 'High Weighted Methods per Class (WMPC)', 'Hub-like Dependency', 'Inappropriate Intimacy', 'Large Class', 'Large Class (SIZE2)', 'Lazy Class', 'Long File', 'Long Method', 'Long Parameter List', 'Message Chains', 'Middle Man', 'Orphan Module', 'Parallel Inheritance Hierarchies', 'Potential Diverge

1438

In [9]:
# ==== Tokenizer + torch datasets ====
tokenizer = create_tokenizer(BACKBONE_NAME, use_fast=True, truncation_side='right')  # right truncation => truncate from end

train_ds = PretokenizedSmellDataset(
    dataset_build.train_examples, tokenizer, dataset_build.label_maps, dataset_build.smell_maps, max_length=MAX_LENGTH
)
val_ds = PretokenizedSmellDataset(
    dataset_build.val_examples, tokenizer, dataset_build.label_maps, dataset_build.smell_maps, max_length=MAX_LENGTH
)
test_ds = PretokenizedSmellDataset(
    dataset_build.test_examples, tokenizer, dataset_build.label_maps, dataset_build.smell_maps, max_length=MAX_LENGTH
)

collator = TokenClassificationCollator(tokenizer=tokenizer, label_pad_id=-100)
train_loader = make_dataloader(train_ds, collator, batch_size=BATCH_SIZE_TRAIN, shuffle=True, num_workers=NUM_WORKERS)
val_loader = make_dataloader(val_ds, collator, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=NUM_WORKERS)
test_loader = make_dataloader(test_ds, collator, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=NUM_WORKERS)


c:\Users\jinmo\miniconda3\envs\yjmd2222-pyexamine\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jinmo\miniconda3\envs\yjmd2222-pyexamine\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jinmo\.cache\huggingface\hub\models--answerdotai--ModernBERT-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an admini

In [10]:
# ==== Optional class weights (for O dominance) ====
class_weights = None
if USE_CLASS_WEIGHTS:
    label_to_id = dataset_build.label_maps.label_to_id
    class_weights = torch.ones(len(label_to_id), dtype=torch.float32)
    if 'O' in label_to_id:
        class_weights[label_to_id['O']] = float(O_WEIGHT)
    print('class_weights:', {k: float(class_weights[v]) for k,v in label_to_id.items()})
else:
    print('class_weights disabled')


class_weights disabled


In [11]:
# ==== Build model (shared encoder + 46 smell-specific heads) ====
model = SharedEncoderSmellHeads.from_hf_pretrained(
    backbone_name=BACKBONE_NAME,
    num_smells=len(dataset_build.smell_maps.smell_to_head),
    num_labels=len(dataset_build.label_maps.label_to_id),
    dropout=DROPOUT,
    trust_remote_code=False,
)
optimizer = build_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)

print('Model ready. hidden_size=', model.hidden_size)


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 710.33it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready. hidden_size= 768


In [12]:
# ==== Train ====
artifacts = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    id_to_label=dataset_build.label_maps.id_to_label,
    label_to_id=dataset_build.label_maps.label_to_id,
    smell_to_head=dataset_build.smell_maps.smell_to_head,
    tokenizer_name_or_path=BACKBONE_NAME,
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    max_grad_norm=MAX_GRAD_NORM,
    amp_enabled=USE_AMP,
    class_weights=class_weights,
    resume_checkpoint_path=RESUME_CHECKPOINT,
    debug_preview_count=3,
)

print('best checkpoint:', artifacts.best_checkpoint_path)
print('last checkpoint:', artifacts.last_checkpoint_path)


C:\Users\jinmo\Documents\Github\encoder-bio\master-thesis-materials\pyexamine\training_inference\train_loop.py:251: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp_enabled and device.type == "cuda"))
c:\Users\jinmo\miniconda3\envs\yjmd2222-pyexamine\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[epoch 0] train_loss=1.4322 val_loss=1.2855 micro_f1=0.5468 non_o_micro_f1=0.6780 seqeval_f1=nan
  truncation report: train 0/462, val 0/585
[epoch 1] train_loss=0.9607 val_loss=1.0163 micro_f1=0.6504 non_o_micro_f1=0.7012 seqeval_f1=nan
  truncation report: train 0/462, val 0/585
[epoch 2] train_loss=0.7420 val_loss=0.7852 micro_f1=0.7582 non_o_micro_f1=0.8425 seqeval_f1=nan
  truncation report: train 0/462, val 0/585
best checkpoint: C:\Users\jinmo\Documents\Github\encoder-bio\master-thesis-materials\pyexamine\training_inference\runs\modernbert_routed_46h\checkpoints\best.pt
last checkpoint: C:\Users\jinmo\Documents\Github\encoder-bio\master-thesis-materials\pyexamine\training_inference\runs\modernbert_routed_46h\checkpoints\epoch_002.pt


In [ ]:
# ==== (Optional) quick test-set evaluation using best checkpoint (manual next step) ====
# You can load the best checkpoint in the inference notebook, or add a test evaluation cell here.
